# Data Basics Notebook
This notebook provides an introduction to working with both the Transfermarkt API
and UnderstatAPIs.

In [1]:
import duckdb
from understatapi import UnderstatClient
from matplotlib import pyplot as plt
import pandas as pd

## Transfermarkt API

In [3]:
# Example: Liverpool players from 2025
q = """
SELECT *
FROM read_csv_auto('https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/players.csv.gz') P
WHERE current_club_domestic_competition_id = 'GB1' AND last_season = 2025
AND P.current_club_id = 31
"""

duckdb.sql(q).show() # Run the query


┌───────────┬────────────┬──────────────┬─────────────────────┬─────────────┬─────────────────┬─────────────────────┬───────────────────────┬────────────────────┬────────────────────────┬─────────────────────┬────────────────────┬────────────┬─────────┬──────────────┬──────────────────────────┬───────────────────┬────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────┬─────────────────────────┬─────────────────────┬─────────────────────────────┐
│ player_id │ first_name │  last_name   │        name         │ last_season │ current_club_id │     player_code     │   country_of_birth    │   city_of_birth    │ country_of_citizenship │    date_of_birth    │    sub_position    │  position  │  foot   │ height_in_cm │ contract_expiration_date │    agent_name     │                                     image_url                                      │      

In [4]:
# Example: liverpool current player valuations over time

q = """
SELECT P.player_id, P.name, V.date, V.market_value_in_eur
FROM read_csv_auto('https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/players.csv.gz') P
JOIN read_csv_auto('https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/player_valuations.csv.gz') V ON P.player_id = V.player_id
WHERE current_club_domestic_competition_id = 'GB1' AND last_season = 2025
AND P.current_club_id = 31
ORDER BY P.player_id, V.date
"""

duckdb.sql(q).show() # Run the query

┌───────────┬─────────────────┬────────────┬─────────────────────┐
│ player_id │      name       │    date    │ market_value_in_eur │
│   int64   │     varchar     │    date    │        int64        │
├───────────┼─────────────────┼────────────┼─────────────────────┤
│    105470 │ Alisson         │ 2012-05-12 │              150000 │
│    105470 │ Alisson         │ 2012-08-02 │              150000 │
│    105470 │ Alisson         │ 2014-07-08 │              300000 │
│    105470 │ Alisson         │ 2015-01-02 │             1000000 │
│    105470 │ Alisson         │ 2015-11-09 │             3500000 │
│    105470 │ Alisson         │ 2017-02-01 │             7000000 │
│    105470 │ Alisson         │ 2017-08-06 │             7000000 │
│    105470 │ Alisson         │ 2018-07-06 │            60000000 │
│    105470 │ Alisson         │ 2019-10-12 │            90000000 │
│    105470 │ Alisson         │ 2020-08-04 │            72000000 │
│       ·   │    ·            │     ·      │                · 

In [5]:
# Example: Basic Performance Stats

q = """
SELECT *
FROM read_csv_auto('https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/appearances.csv.gz') A
LIMIT 10
"""

duckdb.sql(q).show()

┌────────────────┬─────────┬───────────┬────────────────┬────────────────────────┬────────────┬──────────────────┬────────────────┬──────────────┬───────────┬───────┬─────────┬────────────────┐
│ appearance_id  │ game_id │ player_id │ player_club_id │ player_current_club_id │    date    │   player_name    │ competition_id │ yellow_cards │ red_cards │ goals │ assists │ minutes_played │
│    varchar     │  int64  │   int64   │     int64      │         int64          │    date    │     varchar      │    varchar     │    int64     │   int64   │ int64 │  int64  │     int64      │
├────────────────┼─────────┼───────────┼────────────────┼────────────────────────┼────────────┼──────────────────┼────────────────┼──────────────┼───────────┼───────┼─────────┼────────────────┤
│ 2231978_38004  │ 2231978 │     38004 │            853 │                    235 │ 2012-07-03 │ Aurélien Joachim │ CLQ            │            0 │         0 │     2 │       0 │             90 │
│ 2233748_79232  │ 2233748 │  

## Understat API

In [2]:
# Create connection
understat = UnderstatClient()

In [5]:
# Get all player data for a league/season
epl_players = understat.league(league="EPL").get_player_data(season="2025")

# Make into a dataframe
player_df = pd.DataFrame(epl_players)

player_df.head()

,id,player_name,games,time,goals,xG,assists,xA,shots,key_passes,yellow_cards,red_cards,position,team_title,npg,npxG,xGChain,xGBuildup
0,8260,Erling Haaland,29,2439,22,23.13693391531706,7,4.766327390447259,102,21,1,0,F S,Manchester City,19,20.092258475720882,26.238575093448162,4.129314249381423
1,13222,Thiago,31,2662,19,21.101609505712986,1,3.155416488647461,71,19,6,0,F S,Brentford,13,15.773427560925484,19.4751720353961,3.4934082627296448
2,11363,Antoine Semenyo,29,2597,15,11.201724836602807,4,3.091043023392558,68,32,6,0,F M,"Bournemouth,Manchester City",14,9.67938713915646,16.244873739778996,4.950852746143937
3,8272,João Pedro,31,2345,14,14.15835139900446,5,3.9609613977372646,63,28,4,0,F M S,Chelsea,14,14.15835139900446,17.67835896089673,4.659978250041604
4,501,Danny Welbeck,30,1832,12,10.899193301796913,0,1.8765279427170753,43,18,5,0,F S,Brighton,11,8.6156866569072,12.241745796054602,3.454270198009908


In [8]:
player_df.sort_values(by="time", ascending=False).head(20)

,id,player_name,games,time,goals,xG,assists,xA,shots,key_passes,yellow_cards,red_cards,position,team_title,npg,npxG,xGChain,xGBuildup
53,5304,Mikel Merino,21,977,4,4.551519326865673,3,2.2901399955153465,21,12,1,0,F M S,Arsenal,4,4.551519326865673,7.0440468192100525,3.1536439023911953
464,13150,Jack Fletcher,3,97,0,0.037386249750852585,0,0.11487564817070961,1,2,0,0,S,Manchester United,0,0.037386249750852585,0.26276974007487297,0.14789408445358276
212,9501,Fabio Carvalho,6,97,1,0.6895682029426098,0,0,2,0,0,0,M S,Brentford,1,0.6895682029426098,0.6895682029426098,0
294,5355,Jacob Bruun Larsen,23,965,0,1.3876315392553806,1,1.2826955989003181,16,11,0,0,F M S,Burnley,0,1.3876315392553806,4.335738031193614,2.199708631262183
389,10120,Conor Bradley,15,962,0,0.21982916723936796,1,1.1937107481062412,6,4,5,0,D S,Liverpool,0,0.21982916723936796,3.412436105310917,2.4054819140583277
468,13247,Ezra Mayers,6,96,0,0.10989412665367126,0,0.11634047329425812,1,2,0,0,S,West Ham,0,0.10989412665367126,0.39119094237685204,0.16495634242892265
246,13692,Rio Ngumoha,11,95,1,0.546236764639616,0,1.2886440493166447,3,9,0,0,S,Liverpool,1,0.546236764639616,2.188313815742731,0.3534329878166318
351,8489,Jorge Cuenca,14,948,0,0.050821494311094284,0,0.5158752724528313,1,3,5,0,D S,Fulham,0,0.050821494311094284,2.2441382743418217,2.2441382743418217
432,11808,Andrey Santos,22,946,0,1.7154290862381458,0,0.6522208489477634,8,7,4,0,M S,Chelsea,0,1.7154290862381458,5.09560789167881,3.811923023313284
407,10822,Wilson Odobert,24,946,0,2.460178238339722,2,1.9524234533309937,21,20,0,0,F M S,Tottenham,0,2.460178238339722,5.9682222586125135,2.8834201879799366


In [9]:
# Look at all positions

player_df["position"].unique()

array(['F S', 'F M', 'F M S', 'M S', 'F', 'M', 'D M S', 'D S', 'D M', 'D',
       'D F M S', 'S', 'GK', 'GK S'], dtype=object)

In [11]:
# Get all match stats for a particular player - Erling Haaland
player_matches = understat.player(player='8260').get_match_data()

# Convert to dataframe
player_matches_df = pd.DataFrame(player_matches)

# Sort by date
player_matches_df = player_matches_df.sort_values(by="date")

player_matches_df.head()

,goals,shots,xG,time,position,h_team,a_team,h_goals,a_goals,date,id,season,roster_id,xA,assists,key_passes,npg,npxG,xGChain,xGBuildup
190,3,3,1.3227900266647339,33,Sub,Augsburg,Borussia Dortmund,3,5,2020-01-18,12562,2019,389276,0,0,0,3,1.3227900266647339,1.3459999561309814,0.023214099928736687
189,2,3,1.3333300352096558,23,Sub,Borussia Dortmund,FC Cologne,5,1,2020-01-24,12566,2019,389900,0,0,0,2,1.3333300352096558,1.3333300352096558,0
188,2,2,0.7829639911651611,77,FW,Borussia Dortmund,Union Berlin,5,0,2020-02-01,12574,2019,391067,0.06835760176181793,0,1,2,0.7829639911651611,0.7862169742584229,0.5620520114898682
187,0,2,0.10662200301885605,90,FW,Bayer Leverkusen,Borussia Dortmund,4,3,2020-02-08,12584,2019,392772,0,0,0,0,0.10662200301885605,0.422342985868454,0.3633730113506317
186,1,2,0.6940990090370178,80,FW,Borussia Dortmund,Eintracht Frankfurt,4,0,2020-02-14,12592,2019,393399,0.04634920135140419,0,1,1,0.6940990090370178,0.7866899967193604,0.046242501586675644


In [ ]:
# Get all player names and ids and save to csv for later use

leagues = ["EPL", "La_Liga", "Bundesliga", "Serie_A", "Ligue_1"]
seasons = list(range(2015, 2026))

id_to_name = {}
for league in leagues:
    for season in seasons:
        players = understat.league(league=league).get_player_data(season=str(season))
        for p in players:
            id_to_name[p["id"]] = p["player_name"]
            
id_names = pd.DataFrame({"player_id":id_to_name.keys(), "name":id_to_name.values()})
id_names.to_csv("../data/id_names.csv")